In [1]:
!pip install scAnalysis

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 30.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of louvain to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 78.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 87.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 73.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 14.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 44.1 MB/s eta 0:00:00
  Created wheel for louv

In [2]:
!pip install anndata

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 45.5 MB/s eta 0:00:00


In [3]:

import numpy as np

from scAnalysis import (
    sc_io,
    preprocessing,
    quality_control,
    cell_cycle,
    batch_correction,
    dimensionality,
    clustering,
    trajectory,
    differential,
    enrichment,
    visualization,
    interactive_viz,
    imputation,
)

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")

In [4]:
import pandas as pd
import anndata as ad
from scipy.stats import pearsonr
import os

In [7]:
local_h5ad_path = "/content/drive/MyDrive/resources/grn_benchmark/evaluation_data/MSCIC_sc.h5ad"
tf_list_path = "/content/drive/MyDrive/resources/grn_benchmark/prior/tf_all.csv"

In [8]:
data = sc_io.read_h5ad(local_h5ad_path)
data.var.index = sc_io._make_unique(data.var.index.values)
print(f"Loaded: {data.n_obs} cells and {data.n_vars} genes.")

IO: Reading H5AD from '/content/drive/MyDrive/resources/grn_benchmark/evaluation_data/MSCIC_sc.h5ad' ...
IO: Loaded 26,444 cells × 13,431 genes.
Loaded: 26444 cells and 13431 genes.


In [9]:
preprocessing.calculate_qc_metrics(data, qc_vars=["MT-", "mt-"])

quality_control.scrublet(data, verbose=False)
mask_singlets = ~data.obs['predicted_doublet'].astype(bool)
data = data[mask_singlets, :]

data = preprocessing.filter_cells(data, min_genes=200, max_pct_mito=15.0)
data = preprocessing.filter_genes(data, min_cells=3)

filter_cells: keeping 25,445 / 25,459 cells.
filter_genes: keeping 13,431 / 13,431 genes.


In [10]:
preprocessing.normalize_total(data, target_sum=1e4)
preprocessing.log1p(data)

In [11]:
tf_all = pd.read_csv(tf_list_path, header=None)[0].tolist()

available_tfs = [tf for tf in tf_all if tf in data.var.index]
all_genes = data.var.index.tolist()
print(f"Found {len(available_tfs)} Transcription Factors in the dataset.")

Found 1234 Transcription Factors in the dataset.


In [14]:
import scipy.sparse as sp

In [15]:
X_matrix = data.X.toarray() if sp.issparse(data.X) else data.X
n_cells = X_matrix.shape[0]

X_mean = X_matrix.mean(axis=0)
X_std = X_matrix.std(axis=0)
X_std[X_std == 0] = 1e-12

Z_matrix = (X_matrix - X_mean) / X_std

tf_indices = [all_genes.index(tf) for tf in available_tfs]
Z_tf = Z_matrix[:, tf_indices]


corr_matrix = np.dot(Z_tf.T, Z_matrix) / n_cells


weight_matrix = np.abs(corr_matrix)

for i, tf_idx in enumerate(tf_indices):
    weight_matrix[i, tf_idx] = 0.0

print("Extracting top 50,000 edges")

flat_weights = weight_matrix.flatten()
top_k = min(50000, len(flat_weights))

top_indices = np.argpartition(flat_weights, -top_k)[-top_k:]

top_indices = top_indices[np.argsort(flat_weights[top_indices])[::-1]]

tf_idx_2d, gene_idx_2d = np.unravel_index(top_indices, weight_matrix.shape)

edges = []
for i in range(len(top_indices)):
    weight = weight_matrix[tf_idx_2d[i], gene_idx_2d[i]]
    if weight > 0.05:
        edges.append({
            'source': available_tfs[tf_idx_2d[i]],
            'target': all_genes[gene_idx_2d[i]],
            'weight': str(weight)
        })

grn_df = pd.DataFrame(edges)

Extracting top 50,000 edges


In [16]:
output_anndata = ad.AnnData(
    X=np.empty((0, 0)),
    uns={
        "method_id": "scAnalyzer_Pearson",
        "dataset_id": "MSCIC",
        "prediction": grn_df[["source", "target", "weight"]]
    }
)

os.makedirs("output", exist_ok=True)
output_path = "output/MSCIC_scAnalyzer_GRN.h5ad"
output_anndata.write_h5ad(output_path)

In [17]:
print(f"Total number of edges extracted: {len(grn_df)}")

Total number of edges extracted: 50000


In [18]:
!pwd

/content


In [19]:
!git clone --recursive https://github.com/openproblems-bio/task_grn_inference.git

Cloning into 'task_grn_inference'...
remote: Enumerating objects: 85313, done.
remote: Counting objects: 100% (1352/1352), done.
remote: Compressing objects: 100% (728/728), done.
remote: Total 85313 (delta 775), reused 1055 (delta 604), pack-reused 83961 (from 4)
Receiving objects: 100% (85313/85313), 136.22 MiB | 33.79 MiB/s, done.
Resolving deltas: 100% (51005/51005), done.
Submodule 'common' (https://github.com/openproblems-bio/common_resources.git) registered for path 'common'
Cloning into '/content/task_grn_inference/common'...
remote: Enumerating objects: 484, done.        
remote: Counting objects: 100% (233/233), done.        
remote: Compressing objects: 100% (113/113), done.        
remote: Total 484 (delta 174), reused 143 (delta 117), pack-reused 251 (from 1)        
Receiving objects: 100% (484/484), 276.12 KiB | 3.78 MiB/s, done.
Resolving deltas: 100% (246/246), done.
Submodule path 'common': checked out 'f01ff2170161295e89014ee5453c61b29b4e4e77'


In [1]:
%cd /content/task_grn_inference

/content/task_grn_inference


In [23]:
!pip install peekdir

In [25]:
!PeekDir /content/drive/MyDrive/resources/grn_benchmark/inference_data

inference_data/
    300BCG_rna.h5ad
    MSCIC_atac.h5ad
    MSCIC_rna.h5ad
    adamson_rna.h5ad
    nakatake_rna.h5ad
    ... 9 more .h5ad files


In [26]:
!ln -s /content/drive/MyDrive/resources resources

In [27]:
!ls -l resources/grn_benchmark/inference_data/MSCIC_rna.h5ad

-rw------- 1 root root 563042878 May  8 12:12 resources/grn_benchmark/inference_data/MSCIC_rna.h5ad


In [28]:
!pwd

/content/task_grn_inference


In [29]:
!bash scripts/prior/run_consensus.sh \
  --dataset MSCIC \
  --new_model /content/output/MSCIC_scAnalyzer_GRN.h5ad

Config file generated at: src/utils/config.env
Adding new model: /content/output/MSCIC_scAnalyzer_GRN.h5ad
/content/output/MSCIC_scAnalyzer_GRN.h5ad
Running consensus for Regression
Running regression consensus for dataset: MSCIC
{'dataset': 'MSCIC', 'evaluation_data': 'resources/grn_benchmark/inference_data/MSCIC_rna.h5ad', 'regulators_consensus': 'resources/grn_benchmark/prior/regulators_consensus_MSCIC.json', 'predictions': ['/content/output/MSCIC_scAnalyzer_GRN.h5ad']}
Original net shape: (50000, 3)
Supplementary columns for grouping: []
Network shape after cleaning: (50000, 3)
Network shape applying max_n_links: (50000, 3)
Sparsity of /content/output/MSCIC_scAnalyzer_GRN.h5ad: 0.999722825478709
Running consensus for ws distance
Skipping dataset: MSCIC


In [31]:
!pwd

/content/task_grn_inference


In [32]:
!wget -qO miniconda.sh https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!bash ./miniconda.sh -b -f -p /usr/local
!conda --version

PREFIX=/usr/local
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /usr/local
conda 26.3.2


In [ ]:
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

In [ ]:
!conda create -y -p ./genernbi python=3.10
!./genernbi/bin/pip install anndata pandas numpy scipy scikit-learn statsmodels networkx POT pyyaml scanpy lightgbm torch decoupler tqdm_joblib

In [40]:
!pwd

/content/task_grn_inference


In [4]:
!fallocate -l 16G /swapfile
!chmod 600 /swapfile
!mkswap /swapfile
!swapon /swapfile

!free -h

Setting up swapspace version 1, size = 16 GiB (17179865088 bytes)
no label, UUID=ce8bc20c-6f26-4e1a-8c06-655dda0bca69
swapon: /swapfile: swapon failed: Invalid argument
               total        used        free      shared  buff/cache   available
Mem:            12Gi       807Mi       9.2Gi       1.5Gi       2.7Gi        10Gi
Swap:             0B          0B          0B


In [5]:
!env MPLBACKEND=agg bash src/metrics/all_metrics/run_local.sh \
  --dataset MSCIC \
  --prediction /content/output/MSCIC_scAnalyzer_GRN.h5ad \
  --score /content/output/MSCIC_score.h5ad \
  --num_workers 4

Streaming output truncated to the last 5000 lines.


 76%|███████▌  | 1886/2485 [05:13<01:52,  5.31it/s]


 76%|███████▌  | 1888/2485 [05:13<01:28,  6.78it/s]


 76%|███████▌  | 1890/2485 [05:13<01:29,  6.67it/s]


 76%|███████▌  | 1891/2485 [05:13<01:28,  6.69it/s]


 76%|███████▌  | 1893/2485 [05:13<01:21,  7.23it/s]


 76%|███████▌  | 1894/2485 [05:14<01:23,  7.08it/s]


 76%|███████▋  | 1895/2485 [05:14<01:51,  5.28it/s]


 76%|███████▋  | 1897/2485 [05:14<01:20,  7.33it/s]


 76%|███████▋  | 1899/2485 [05:14<01:22,  7.13it/s]


 76%|███████▋  | 1901/2485 [05:14<01:05,  8.98it/s]


 77%|███████▋  | 1903/2485 [05:15<01:14,  7.77it/s]


 77%|███████▋  | 1905/2485 [05:15<01:14,  7.79it/s]


 77%|███████▋  | 1907/2485 [05:15<01:29,  6.48it/s]


 77%|███████▋  | 1908/2485 [05:16<01:24,  6.86it/s]


 77%|███████▋  | 1910/2485 [05:16<01:20,  7.12it/s]


 77%|███████▋  | 1912/2485 [05:16<01:11,  8.06it/s]


 77%|███████▋  | 1914/2485 [05:16<01:20,  7.09it/s]


 77%|███████▋  | 1917/2485 [0

In [6]:
import anndata as ad
import pandas as pd

score_data = ad.read_h5ad("/content/output/MSCIC_score.h5ad")
scores_dict = score_data.uns.get('metrics', score_data.uns)


results_df = pd.DataFrame(list(scores_dict.items()), columns=['Metric', 'Score'])

print(results_df.to_string(index=False))

       Metric                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               Score
   dataset_id                                                                                                                                                                                                                                                                                                                         

In [12]:
import anndata as ad
import pandas as pd
from IPython.display import display

score_data = ad.read_h5ad("../output/MSCIC_score.h5ad")

metric_ids = score_data.uns['metric_ids']
metric_values = score_data.uns['metric_values']

df_scores = pd.DataFrame({
    'Metric': metric_ids,
    'Score (scAnalyzer)': metric_values
})


df_scores['Score (scAnalyzer)'] = df_scores['Score (scAnalyzer)'].round(3)


df_scores = df_scores.sort_values(by='Score (scAnalyzer)', ascending=False).reset_index(drop=True)


display(df_scores)

df_scores.to_csv("../output/scAnalyzer_MSCIC_results.csv", index=False)

,Metric,Score (scAnalyzer)
0,reactome_2022_gs_n_active,893.0
1,bioplanet_2019_gs_n_active,816.0
2,hallmark_2020_gs_n_active,32.0
3,wikipathways_2019_gs_n_active,266.0
4,kegg_2021_gs_n_active,215.0
5,go_bp_2023_gs_n_active,1965.0
6,wikipathways_2019_gs_precision,0.9518072289156626
7,kegg_2021_gs_precision,0.9354838709677419
8,bioplanet_2019_gs_precision,0.8908296943231441
9,reactome_2022_gs_precision,0.8450704225352113


In [8]:
!zip -r mscic_output.zip ../output

  adding: ../output/ (stored 0%)
  adding: ../output/MSCIC_scAnalyzer_GRN.h5ad (deflated 82%)
  adding: ../output/MSCIC_score.h5ad (deflated 89%)
